In [1]:
# Cell 1: Setup & Configuration
import os
import pickle
import pandas as pd
import numpy as np
import re
from gensim.models import Word2Vec
from tqdm.notebook import tqdm

# Configuration Paths
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
HIDDEN_DIM = 128
NUM_ITEMS = 22884

ImportError: DLL load failed while importing _rbfinterp_pythran: An Application Control policy has blocked this file.

In [ ]:
# Cell 2: Load & Merge Datasets
print("Loading Mappings and Metadata...")
with open('C:/Users/Rajdeep Kumar/movie-recsys/data/processed/mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)
    movie_to_idx = mappings['movie_to_idx']

movies_df = pd.read_csv('C:/Users/Rajdeep Kumar/movie-recsys/data/raw/movie.csv')
tmdb_df = pd.read_csv('C:/Users/Rajdeep Kumar/movie-recsys/data/processed/tmdb_features.csv')

# Load tags if available
try:
    tags_df = pd.read_csv('C:/Users/Rajdeep Kumar/movie-recsys/data/raw/tag.csv')
    tags_grouped = tags_df.groupby('movieId')['tag'].apply(lambda x: ' '.join(str(v).lower() for v in x)).reset_index()
    df = pd.merge(movies_df, tags_grouped, on='movieId', how='left')
except FileNotFoundError:
    print("⚠️ tag.csv not found, proceeding with genres only...")
    df = movies_df.copy()
    df['tag'] = ""

# Merge TMDB data
master_df = pd.merge(df, tmdb_df, on='movieId', how='left').fillna("")
print(f"Total movies loaded: {len(master_df)}")
master_df.head(3)

Loading Mappings and Metadata...
Total movies loaded: 27278


,movieId,title,genres,tag,overview,actors,director
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,watched computer animation disney animated fea...,"Led by Woody, Andy's toys live happily in his ...",TomHanks TimAllen DonRickles,JohnLasseter
1,2,Jumanji (1995),Adventure|Children|Fantasy,time travel adapted from:book board game child...,When siblings Judy and Peter discover an encha...,RobinWilliams KirstenDunst BradleyPierce,JoeJohnston
2,3,Grumpier Old Men (1995),Comedy|Romance,old people that is actually funny sequel fever...,A family wedding reignites the ancient feud be...,WalterMatthau JackLemmon Ann-Margret,HowardDeutch


In [ ]:
# Cell 3: Sentence Transformer Encoding + PCA Dimensionality Reduction
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import torch

print("🚀 Loading lightweight Transformer model (all-MiniLM-L6-v2)...")
# Automatically snap to the GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
#model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)

print("🧠 Formatting Natural Language Corpus...")
transformer_corpus = []
movie_id_list = []

for idx, row in tqdm(master_df.iterrows(), total=len(master_df)):
    movieId = row['movieId']
    
    genres = str(row['genres']).replace('|', ', ')
    tags = str(row['tag'])
    actors = str(row.get('actors', '')).replace(' ', ', ')
    director = str(row.get('director', ''))
    overview = str(row.get('overview', ''))
    
    # Format into a highly structured, readable paragraph for the attention mechanism
    combined_text = f"Genres: {genres}. Tags: {tags}. Cast: {actors}. Director: {director}. Plot: {overview}"
    
    transformer_corpus.append(combined_text)
    movie_id_list.append(movieId)

print("\n⚡ Encoding 27,000+ movie plots (Blazing fast on CUDA)...")
# Encode to 384-dimensional dense vectors
raw_embeddings = model.encode(transformer_corpus, show_progress_bar=True, batch_size=256, device=device)

print("\n📉 Reducing dimensions from 384 to 128 to match SASRec architecture...")
# Compress the semantic data to fit the 128-dim hidden layer
pca = PCA(n_components=HIDDEN_DIM)
reduced_embeddings = pca.fit_transform(raw_embeddings)

print("💉 Mapping Transformer vectors to SASRec Token IDs...")
pretrained_matrix = np.random.normal(scale=0.01, size=(NUM_ITEMS + 1, HIDDEN_DIM)).astype(np.float32)
pretrained_matrix[0] = 0.0 # Zero out padding

found_count = 0
for i, movie_id in enumerate(movie_id_list):
    if movie_id in movie_to_idx:
        token_id = movie_to_idx[movie_id]
        pretrained_matrix[token_id] = reduced_embeddings[i]
        found_count += 1
        
save_path = r"C:/Users/Rajdeep Kumar/movie-recsys/data/processed/transformer_embeddings.npy"
np.save(save_path, pretrained_matrix)

print(f"\n✅ Pre-trained Transformer vectors mapped for {found_count} out of {NUM_ITEMS} items.")
print(f"💾 Highly Enriched Transformer Matrix saved to {save_path}")

🚀 Loading lightweight Transformer model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

🧠 Formatting Natural Language Corpus...


  0%|          | 0/27278 [00:00<?, ?it/s]


⚡ Encoding 27,000+ movie plots (Blazing fast on CUDA)...


Batches:   0%|          | 0/107 [00:00<?, ?it/s]


📉 Reducing dimensions from 384 to 128 to match SASRec architecture...
💉 Mapping Transformer vectors to SASRec Token IDs...

✅ Pre-trained Transformer vectors mapped for 22884 out of 22884 items.
💾 Highly Enriched Transformer Matrix saved to C:/Users/Rajdeep Kumar/movie-recsys/data/processed/transformer_embeddings.npy
